In [ ]:
import pandas as pd 
import polars as pl 
from pathlib import Path
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace
from datetime import datetime

from sqlalchemy.engine import Engine

import time
import gc

from logistics_helper_functions import format_time_period, run_logistics_nd_aggregations_lvl5, calculer_indicateur_lvl5
from pnlp_utils import log_msg, push_dataframe_to_db, check_partition_exist, upsert_partition, create_partition_indexes

In [ ]:
# The denominator in these indicators is set to 1 (dummy fixed value) after building the num/den per period.

In [ ]:
# Parameters
YEAR = 2026

In [ ]:
coc_selection = [
    "Stock Disponible Utilisable-SDU",
    "MSD",
    "CMM"
]

columns_expected = [    
    'aggregation_temporelle',
    'period',
    'annee',    
    'indicateur_name',
    'de_name',
    'province',
    'zone_de_sante',
    'aire_de_sante',
    'fosa',
    "fosa_dhis2_id",
    'absolute_indicateur_value',
    'taux_indicateur_value',
    'indicateur_num',
    'indicateur_den'
]


### Start processing

In [ ]:
# table names
target_table = "logistics_ND_table_produits_traceurs_fosa" 

# Database Connection 
engine = create_engine(workspace.database_url)
print(f"Connected to: {engine.url.host}")

# Load available years
target_years = [YEAR]
# target_years = [2024, 2025, 2026]
print(f"Running for years: {target_years}")

In [ ]:
for target_year in target_years:
    
    log_msg(f"***Running for year: {target_year}")
    try:
        logistics_path = Path("~/workspace/pnlp-tdb-pipeline/data/interim") / f"PNLP_logistics_{target_year}.feather"
        logistics_level_5 = pl.read_ipc(logistics_path, memory_map=False)
        logistics_level_5 = logistics_level_5.filter((pl.col("COC_NAME").is_in(coc_selection)))
    except Exception as e:
        log_msg(f"Error loading {logistics_path}. No data found, year skipped.", "error")
        continue
    
    # Format -> Following original R scripts
    df_logistics_mensuel = format_time_period(logistics_level_5)  

    # 4.14.1. % des FOSA n'ayant pas eu de rupture de stocks 
    log_msg(f"Computing logistics produits traceurs FOSA..")
    df_stock_disp = calculer_indicateur_lvl5(df_logistics_mensuel, indicateur_name="Stock_Disp")
    df_mois_stock = calculer_indicateur_lvl5(df_logistics_mensuel, indicateur_name="Mois_stock")
    df_cmm = calculer_indicateur_lvl5(df_logistics_mensuel, indicateur_name="CMM")
    
    df_produits_traceurs = pl.concat([df_stock_disp, df_mois_stock, df_cmm])
    
    # clean
    del logistics_level_5, df_logistics_mensuel, df_stock_disp, df_mois_stock, df_cmm
    collected = gc.collect()

    # Run temporal aggregations
    df_year_agg = run_logistics_nd_aggregations_lvl5(df_produits_traceurs)
    assert df_year_agg.height == df_year_agg.unique().height, "Duplicated values found!" # quick check!
    
    print("Check expected columns..", end=" ")
    df_year_agg = df_year_agg.select(columns_expected)
    print("OK!")

    # filter by "Mois"
    df_year_agg = df_year_agg.filter(pl.col("aggregation_temporelle") == "Mois")
    
    # Rita prefer these names ;) 
    df_year_agg = df_year_agg.with_columns(
        pl.col("indicateur_name").replace(
            {
                "Mois Stock": "Mois de Stock Disponible",
                "Stock Disp": "Stock Disponible Utilisable",
                "Cmm": "Consommation Mensuelle Moyenne"
            }
        )
    )

    log_msg(f"Pushing logistics rupture FOSA to db. Target table: {target_table}")
    tmp_table = f"{target_table}_tmp" 
    push_dataframe_to_db(
        df=df_year_agg, 
        table_name=tmp_table, 
        engine=engine, 
        push_behaviour="replace",
        chunksize=400000
    )
    
    # update partition
    upsert_partition(engine, target_table, tmp_table, target_year)
    
    log_msg("Creating indexes..")
    index_columns = ["province", "zone_de_sante", "aire_de_sante", "period"]
    create_partition_indexes(engine, target_table, target_year, index_columns)
    
    # clean
    del df_produits_traceurs, df_year_agg, index_columns
    collected = gc.collect()
    print(collected)
